In [ ]:
import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections
from nighthawk.models.valuation import node_price_predictor



# --- config ---
start_date    = '2026-01-10'
end_date      = '2026-03-30'
run_number    = 1
opexchange    = 'SPP'

if int(run_number) == 1:
    data_location = 'training'
else:
    data_location = 'secondRun'

work_str = '''
import pandas as pd
import numpy as np
import time
import sys
import pickle
import dill
import warnings
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
print("Here the job starts!")
print(str(datetime.now()))

bucket_name      = sys.argv[1]
save_file_folder = sys.argv[2]
job_table_id     = sys.argv[3]
with open("job_info.pickle", "rb") as file:
    job_info = dill.load(file)

job_id    = job_info["jobId"]
record_id = job_info["recordId"]
node_num  = job_info["node_num"]
dt        = job_info["dt"]
# **********************do not change code above**********************************

import sys
try:
    sys.path.append("/var/www/python/Prod/nighthawk/")
except:
    print("oh..this is in kubernetes")

from nighthawk.models.valuation import ve_model_functions

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

opexchange     = "SPP"
gs_loc         = "gs://ve_fourier/production/SPP/''' + data_location + '''"

max_retries = 3
retry_delay = 15
for attempt in range(max_retries):
    try:
        data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
        data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt < max_retries - 1:
            time.sleep(retry_delay)
        else:
            raise

selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
data_df = data_df[selected_columns]

feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
if dt not in feb2021_dates:
    data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

if "dtHr" not in data_df.columns:
    data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


class DNN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
        super(DNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size_2, output_size)
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu2(self.fc2(out))
        return self.fc3(out)


class QuantileDNN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
        super(QuantileDNN, self).__init__()
        self.quantiles = quantiles
        self.output_size = output_size
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu2(self.fc2(out))
        x = self.fc3(out)
        return x.view(x.size(0), self.output_size, len(self.quantiles))


class QuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super(QuantileLoss, self).__init__()
        self.quantiles = quantiles
    def forward(self, predictions, targets):
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i]
            losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
        return torch.mean(torch.cat(losses, dim=2))


def predict_and_collect(model, data_loader, criterion, scaler_y=None):
    model.eval()
    total_loss = 0.0
    predictions = []
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, labels).item()
            predictions.append(outputs.cpu().numpy())
    predictions = np.concatenate(predictions, axis=0)
    if normalize and scaler_y is not None:
        if predictions.ndim == 3:
            num_samples, num_outputs, num_quantiles = predictions.shape
            transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
            transposed = scaler_y.inverse_transform(transposed)
            predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
        else:
            predictions = scaler_y.inverse_transform(predictions)
    return predictions, total_loss / len(data_loader)


quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
quantile_names = [f"q{int(q * 100)}" for q in quantiles]
hidden_size_1  = 32
hidden_size_2  = 8
learning_rate  = 0.0001
num_epochs     = 200
normalize      = True
device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mean_criterion     = nn.MSELoss()
quantile_criterion = QuantileLoss(quantiles)

valuation_models = pd.DataFrame()

for y_var in [["da_total", "rt_total"]]:
    trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
    xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
        opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
        trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

    xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/5, random_state=42)
    scaler_x, scaler_y = None, None
    if normalize:
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
        xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
        xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
        ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
        yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
        ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
    X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
    Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
    X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
    Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
    X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
    Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)
    train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True)
    validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
    test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
    input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
    mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
    quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
    mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
    quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

    for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                         (quantile_model, quantile_optimizer, quantile_criterion)]:
        best_val_loss, patience, no_improve = float("inf"), 5, 0
        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
            model.eval()
            val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
            if val_loss < best_val_loss:
                best_val_loss, no_improve = val_loss, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break


    mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
    quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)

    if normalize and scaler_y is not None:
        ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

    mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
    q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
    quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

    result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
    result["dtHr"] = pd.to_datetime(result["dtHr"])
    result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
    result["hr"]   = result["dtHr"].dt.hour + 1
    valuation_models = pd.concat([valuation_models, result])

valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
            [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

df  = valuation_models


testing_data_file_name = "gs://" + bucket_name + "/" + save_file_folder + "/record_" + str(record_id) + "/prediction/jobId_" + str(job_id) + ".csv"
df.to_csv(testing_data_file_name, index=False)
print("Here the job ends!")
print(str(datetime.now()))
'''

all_pred_dfs= []
# --- pull node list ---
conn = connections.get_sql_connection(database='Darwin_' + opexchange.upper())
daily_node_df = pd.read_sql(
    f"SELECT DISTINCT dt, node_num FROM Fourier_{opexchange}.nodeSelection WHERE dt >= '{start_date}' and dt <= '{end_date}'", conn)
daily_node_df['dt'] = daily_node_df['dt'].astype(str)
daily_node_df = daily_node_df
if daily_node_df.empty:
    print('  No nodes found, skipping.')
print(f'  {len(daily_node_df)} nodes')

npp = node_price_predictor.NodePricePredictor(opexchange, daily_node_df[['dt', 'node_num']])
print('  record_id', npp.get_record_id())

yaml_dict = {'spec': {'parallelism': 800, 'template': {'spec': {
    'containers': [{'resources': {
        'limits':   {'cpu': '1.5', 'memory': '11Gi'},
        'requests': {'cpu': '1.5', 'memory': '11Gi'}
    }}],
    'nodeSelector': {'cloud.google.com/gke-nodepool': 've-pool-e2hm4-spot'}}}}}

predict_table = npp.run(work_str, envir='KUBERNETES', scale='FULL',
                        initialization='us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn',
                        yaml_dict_original=yaml_dict,
                        install_basic_pkgs=False)
print('  predict_table:', predict_table)

all_pred_dfs.append(bigquery_functions.download_df_from_bq(f"SELECT * FROM `{predict_table[0]}`"))
df = pd.concat(all_pred_dfs, ignore_index=True) if all_pred_dfs else pd.DataFrame()
df.to_csv('/var/www/python/Qingcheng/WFiles/sppfourier_with_dayzer.csv', index=False)
print(f'Saved to spptest.csv: {df.shape}')


In [ ]:
import sys
try:
    sys.path.append("/var/www/python/Prod/nighthawk/")
except:
    print("oh..this is in kubernetes")

from nighthawk.models.valuation import ve_model_functions

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

node_num = 656
dt = '2026-03-02'

opexchange     = "SPP"
gs_loc         = "gs://ve_fourier/production/SPP/''' + data_location + '''"

max_retries = 3
retry_delay = 15
for attempt in range(max_retries):
    try:
        data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
        data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt < max_retries - 1:
            time.sleep(retry_delay)
        else:
            raise

selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
data_df = data_df[selected_columns]

feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
if dt not in feb2021_dates:
    data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

if "dtHr" not in data_df.columns:
    data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


class DNN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
        super(DNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size_2, output_size)
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu2(self.fc2(out))
        return self.fc3(out)


class QuantileDNN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
        super(QuantileDNN, self).__init__()
        self.quantiles = quantiles
        self.output_size = output_size
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu2(self.fc2(out))
        x = self.fc3(out)
        return x.view(x.size(0), self.output_size, len(self.quantiles))


class QuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super(QuantileLoss, self).__init__()
        self.quantiles = quantiles
    def forward(self, predictions, targets):
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i]
            losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
        return torch.mean(torch.cat(losses, dim=2))


def predict_and_collect(model, data_loader, criterion, scaler_y=None):
    model.eval()
    total_loss = 0.0
    predictions = []
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, labels).item()
            predictions.append(outputs.cpu().numpy())
    predictions = np.concatenate(predictions, axis=0)
    if normalize and scaler_y is not None:
        if predictions.ndim == 3:
            num_samples, num_outputs, num_quantiles = predictions.shape
            transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
            transposed = scaler_y.inverse_transform(transposed)
            predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
        else:
            predictions = scaler_y.inverse_transform(predictions)
    return predictions, total_loss / len(data_loader)


quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
quantile_names = [f"q{int(q * 100)}" for q in quantiles]
hidden_size_1  = 32
hidden_size_2  = 8
learning_rate  = 0.0001
num_epochs     = 200
normalize      = True
device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mean_criterion     = nn.MSELoss()
quantile_criterion = QuantileLoss(quantiles)

valuation_models = pd.DataFrame()

for y_var in [["da_total", "rt_total"]]:
    trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
    xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
        opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
        trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

    xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/5, random_state=42)
    scaler_x, scaler_y = None, None
    if normalize:
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
        xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
        xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
        ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
        yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
        ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
    X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
    Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
    X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
    Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
    X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
    Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)
    train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True)
    validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
    test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
    input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
    mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
    quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
    mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
    quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

    for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                         (quantile_model, quantile_optimizer, quantile_criterion)]:
        best_val_loss, patience, no_improve = float("inf"), 5, 0
        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
            model.eval()
            val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
            if val_loss < best_val_loss:
                best_val_loss, no_improve = val_loss, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break


    mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
    quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)

    if normalize and scaler_y is not None:
        ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

    mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
    q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
    quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

    result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
    result["dtHr"] = pd.to_datetime(result["dtHr"])
    result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
    result["hr"]   = result["dtHr"].dt.hour + 1
    valuation_models = pd.concat([valuation_models, result])

valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
            [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

df  = valuation_models


testing_data_file_name = "gs://" + bucket_name + "/" + save_file_folder + "/record_" + str(record_id) + "/prediction/jobId_" + str(job_id) + ".csv"
df.to_csv(testing_data_file_name, index=False)
print("Here the job ends!")
print(str(datetime.now()))